# L1 Grounding Ambiguity Multi-Hop Dataset
## CLUTRR + HotpotQA + 2WikiMultihopQA — 600 Examples

This notebook demonstrates the unified dataset for studying **L1 grounding ambiguity** in multi-hop reasoning.

Three datasets are standardised to a common schema (`exp_sel_data_out`):
- **clutrr_v1** (200 examples): CLUTRR kinship-chain reasoning, hops 3–8 — `l1_ambiguity=none`
- **hotpotqa** (200 examples): HotpotQA Wikipedia distractor multi-hop QA — `l1_ambiguity=low`
- **2wikimultihopqa** (200 examples): bridge + comparison question types — `l1_ambiguity=medium/high`

All examples share the schema: `input`, `output`, `metadata_id`, `metadata_fold`, `metadata_domain`, `metadata_hop_count`, `metadata_l1_ambiguity_level`, `metadata_gold_atomic_facts`.

In [1]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru is NOT pre-installed on Colab — always install
_pip('loguru==0.7.3')

# Core packages — pre-installed on Colab, install locally only to match Colab env
if 'google.colab' not in sys.modules:
    _pip('matplotlib==3.10.0')


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [2]:
import ast
import json
import os
import random
import sys
from pathlib import Path

import matplotlib
matplotlib.use('Agg')  # headless-safe backend
import matplotlib.pyplot as plt
from loguru import logger

logger.remove()
logger.add(sys.stdout, level='INFO', format='{time:HH:mm:ss}|{level:<7}|{message}')

1

In [3]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-70ae4e-the-empty-environment-test-calibration-f/main/round-2/dataset-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists('mini_demo_data.json'):
        with open('mini_demo_data.json') as f:
            return json.load(f)
    raise FileNotFoundError('Could not load mini_demo_data.json')

In [4]:
data = load_data()
logger.info(f"Loaded {data['metadata']['total_examples']} examples from {len(data['datasets'])} datasets")

06:19:13|INFO   |Loaded 600 examples from 3 datasets


## Config

Tunable parameters. Minimum values shown here for fast demo; original production values commented out.

In [5]:
RANDOM_SEED = 42        # reproducibility seed
N_PER_DATASET = 3       # examples to use per dataset (original: 200)
HOP_MIN = 3             # minimum hop count filter for CLUTRR (original: 3)
HOP_MAX = 8             # maximum hop count filter for CLUTRR (original: 8)
TRAIN_FRAC = 0.70       # train split fraction (original: 0.70)
VAL_FRAC = 0.15         # val split fraction (original: 0.15)
# test_frac = 1 - TRAIN_FRAC - VAL_FRAC = 0.15

random.seed(RANDOM_SEED)

## Dataset Structure

Each dataset entry contains a list of examples. Each example has the `exp_sel_data_out` schema fields.

In [6]:
# Flatten all examples from all datasets
all_examples = []
for ds in data["datasets"]:
    all_examples.extend(ds["examples"])

logger.info(f"Total examples in mini demo: {len(all_examples)}")
logger.info(f"Schema keys: {list(all_examples[0].keys())}")

06:19:13|INFO   |Total examples in mini demo: 9


06:19:13|INFO   |Schema keys: ['input', 'output', 'metadata_id', 'metadata_fold', 'metadata_domain', 'metadata_hop_count', 'metadata_l1_ambiguity_level', 'metadata_gold_atomic_facts']


## Fold Assignment

The `assign_fold` function maps example index to train/val/test split based on fractional position.

In [7]:
def assign_fold(idx: int, total: int) -> str:
    r = idx / total
    if r < 0.70:
        return "train"
    elif r < 0.85:
        return "val"
    return "test"


# Verify fold distribution in the loaded data
fold_counts = {}
for ex in all_examples:
    fold = ex["metadata_fold"]
    fold_counts[fold] = fold_counts.get(fold, 0) + 1

logger.info(f"Fold distribution: {fold_counts}")

06:19:13|INFO   |Fold distribution: {'train': 9}


## KINSHIP_LABELS Mapping

The integer-to-relation-name mapping used to decode CLUTRR target labels.

In [8]:
KINSHIP_LABELS = {
    0: "daughter", 1: "son", 2: "granddaughter", 3: "grandson",
    4: "mother", 5: "father", 6: "grandmother", 7: "grandfather",
    8: "sister", 9: "brother", 10: "niece", 11: "nephew",
    12: "aunt", 13: "uncle", 14: "wife", 15: "husband",
    16: "great-grandmother", 17: "great-grandfather",
}

# Show the distribution of answers in CLUTRR examples
clutrr_examples = [ex for ex in all_examples if ex["metadata_id"].startswith("clutrr")]
answer_dist = {}
for ex in clutrr_examples:
    ans = ex["output"]
    answer_dist[ans] = answer_dist.get(ans, 0) + 1

logger.info(f"CLUTRR answer distribution: {answer_dist}")

06:19:13|INFO   |CLUTRR answer distribution: {'daughter': 1, 'granddaughter': 1, 'father': 1}


## Gold Atomic Facts

Each example stores `metadata_gold_atomic_facts` as a JSON-encoded list of `{predicate, args, span_start, span_end}` dicts.

In [9]:
# Compute statistics on gold atomic facts
fact_counts = []
for ex in all_examples:
    facts = json.loads(ex["metadata_gold_atomic_facts"])
    fact_counts.append(len(facts))

avg_facts = sum(fact_counts) / len(fact_counts) if fact_counts else 0
logger.info(f"Avg gold atomic facts per example: {avg_facts:.1f}  (min={min(fact_counts)}, max={max(fact_counts)})")

# Show one example in full detail
sample = all_examples[0]
logger.info(f"Sample id: {sample['metadata_id']}")
logger.info(f"  domain: {sample['metadata_domain']}  hops: {sample['metadata_hop_count']}  l1_ambiguity: {sample['metadata_l1_ambiguity_level']}")
logger.info(f"  output: {sample['output']}")

06:19:13|INFO   |Avg gold atomic facts per example: 2.3  (min=0, max=4)


06:19:13|INFO   |Sample id: clutrr_v1_0


06:19:13|INFO   |  domain: kinship_reasoning  hops: 4  l1_ambiguity: none


06:19:13|INFO   |  output: daughter


## Dataset Statistics

Summary statistics across the full dataset (from `data['metadata']`), plus distributions from the loaded examples.

In [10]:
# Compute per-dataset counts and l1 ambiguity distribution
by_dataset = {}
by_l1 = {}
by_hop = {}

for ex in all_examples:
    ds_name = ex["metadata_id"].split("_")[0]
    by_dataset[ds_name] = by_dataset.get(ds_name, 0) + 1

    l1 = ex["metadata_l1_ambiguity_level"]
    by_l1[l1] = by_l1.get(l1, 0) + 1

    hops = ex["metadata_hop_count"]
    by_hop[hops] = by_hop.get(hops, 0) + 1

logger.info(f"By dataset: {by_dataset}")
logger.info(f"By l1 ambiguity: {by_l1}")
logger.info(f"By hop count: {by_hop}")

06:19:13|INFO   |By dataset: {'clutrr': 3, 'hotpotqa': 3, '2wiki': 3}


06:19:13|INFO   |By l1 ambiguity: {'none': 3, 'low': 3, 'high': 2, 'medium': 1}


06:19:13|INFO   |By hop count: {4: 1, 3: 5, 2: 3}


## Visualisation

Three bar charts: examples per dataset, L1 ambiguity level distribution, and hop-count distribution.

In [11]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("L1 Grounding Ambiguity Multi-Hop Dataset", fontsize=13)

# Panel 1 — examples per dataset
ax = axes[0]
ax.bar(list(by_dataset.keys()), list(by_dataset.values()), color=["#4e79a7", "#f28e2b", "#e15759"])
ax.set_title("Examples per Dataset")
ax.set_xlabel("Dataset")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=15)

# Panel 2 — L1 ambiguity level
ax = axes[1]
l1_order = ["none", "low", "medium", "high"]
l1_vals = [by_l1.get(k, 0) for k in l1_order]
colors = ["#59a14f", "#f28e2b", "#e15759", "#b07aa1"]
ax.bar(l1_order, l1_vals, color=colors)
ax.set_title("L1 Ambiguity Level")
ax.set_xlabel("Level")
ax.set_ylabel("Count")

# Panel 3 — hop count distribution
ax = axes[2]
hop_keys = sorted(by_hop.keys())
hop_vals = [by_hop[k] for k in hop_keys]
ax.bar([str(k) for k in hop_keys], hop_vals, color="#76b7b2")
ax.set_title("Hop Count Distribution")
ax.set_xlabel("Hop Count")
ax.set_ylabel("Count")

plt.tight_layout()
plt.savefig("dataset_stats.png", dpi=80, bbox_inches="tight")
plt.show()
print("Saved dataset_stats.png")

# Summary table
print(f"\n{'='*50}")
print(f"{'Metric':<30} {'Value':>10}")
print(f"{'='*50}")
print(f"{'Total examples (full)':<30} {data['metadata']['total_examples']:>10}")
print(f"{'Mini demo examples':<30} {len(all_examples):>10}")
print(f"{'Datasets':<30} {len(data['datasets']):>10}")
for ds_name, cnt in by_dataset.items():
    print(f"  {ds_name:<28} {cnt:>10}")
print(f"{'L1 ambiguity levels':<30} {len(by_l1):>10}")
for level in l1_order:
    if level in by_l1:
        print(f"  {level:<28} {by_l1[level]:>10}")
print(f"{'Avg gold atomic facts':<30} {avg_facts:>10.1f}")
print(f"{'='*50}")

Saved dataset_stats.png

Metric                              Value
Total examples (full)                 600
Mini demo examples                      9
Datasets                                3
  clutrr                                3
  hotpotqa                              3
  2wiki                                 3
L1 ambiguity levels                     4
  none                                  3
  low                                   3
  medium                                1
  high                                  2
Avg gold atomic facts                 2.3


/tmp/claude-1000/claude-1000/ipykernel_3432893/2057968336.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
